# GPU 长什么样：lspci 与 PCIe 拓扑

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-02-gpu-on-pcie.ipynb)

从系统总线视角看 GPU。用 lspci、nvidia-smi -q 和 /proc/cpuinfo 把 GPU 和它在主板上的位置、NUMA 归属联系起来，为后面理解多卡、PCIe 带宽和拓扑调度打基础。

本 notebook 配合教程章节 [GPU 长什么样：lspci 与 PCIe 拓扑](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

[Lab 01](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-01-meet-your-gpu/) 里 `nvidia-smi` 显示了一行 `00000000:00:04.0`，那是 GPU 的 **PCIe 总线地址**。在单卡 Colab 里它只是个数字，但在多卡服务器里，GPU 挂在哪个 NUMA 节点、走哪条 PCIe Switch、和谁共享带宽，直接决定了训练和推理的性能。Kubernetes 的拓扑调度、HAMi 的设备分配，本质上都在围绕这些信息做决策。本 Lab 用最小命令把这些概念落地。

先建立整体图景：CPU 和离它最近的内存组成一个 **NUMA 节点**；CPU 里的 PCIe 根复合体通过一条 **PCIe 链路**连到 GPU，链路的代际和 lane 数决定了 **带宽** 上限；`lspci` 等命令则分别观察这条通路上的不同部分。下图把这几个概念和本 Lab 用到的命令对应起来：

![一块 GPU 在机器里的位置：NUMA、PCIe、带宽与查看命令](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/pcie-concept-map.svg)

## 第 1 步：用 lspci 看总线

Colab 镜像没有预装 `lspci`，直接运行会报 `lspci: command not found`。先用 `apt-get` 安装它所在的 `pciutils` 包（不需要先 `apt-get update`，约 10 秒），再过滤出 NVIDIA 设备：

In [ ]:
!apt-get install -y -qq pciutils > /dev/null
!lspci | grep -i nvidia

预期输出类似：

```text
00:04.0 3D controller: NVIDIA Corporation TU104GL [Tesla T4] (rev a1)
```

读法：

- `00:04.0` = **总线 : 设备 . 功能**（bus:device.function），和 `nvidia-smi` 里的 `00:04.0` 对应；
- `3D controller` = 设备类别（数据中心卡常显示为 `3D controller` 而非 `VGA compatible controller`，因为它没有显示输出）；
- `TU104GL` = 芯片代号（Turing）。

## 第 2 步：用 nvidia-smi 查 PCIe 链路

In [ ]:
!nvidia-smi --query-gpu=pci.bus_id,pcie.link.gen.max,pcie.link.gen.current,pcie.link.width.max,pcie.link.width.current --format=csv

预期看到：

```text
pci.bus_id, pcie.link.gen.max, pcie.link.gen.current, pcie.link.width.max, pcie.link.width.current
00000000:00:04.0, 3, 1, 16, 16
```

关键字段：

- `pcie.link.gen.max: 3` = 设备支持的最高代际 PCIe **Gen3**（T4 上限）；
- `pcie.link.gen.current: 1` = **当前**运行在 Gen1。这不是故障：GPU 空载时会把链路降速省电，有数据传输时自动升回 Gen3；
- `pcie.link.width.max: 16` = 16 条 lane，即 **x16**。

想看完整的 PCI 小节，可以从 `nvidia-smi -q` 的完整报告里截取。注意 `-d` 参数不接受 `PCI`（580 驱动会报 `Failed to parse --display/-d flags`），所以用 `sed` 按小节标题截取：

In [ ]:
!nvidia-smi -q | sed -n "/^    PCI$/,/^    Fan Speed/p"

它会把上面这些字段按 `GPU Link Info → PCIe Generation / Link Width` 的层级展开，还会列出 Device Id（T4 是 `0x1EB810DE`，`10DE` 是 NVIDIA 的 vendor id）。

## 第 3 步：算一下理论带宽

PCIe 是全双工的，每条 lane 单向带宽约：



| 代际 | 单 lane 单向 (GB/s) | x16 单向 (GB/s) | x16 双向合计 (GB/s) |
| :--- | :--- | :--- | :--- |
| Gen3 | ~0.985 | ~15.75 | ~31.5 |
| Gen4 | ~1.969 | ~31.5 | ~63 |
| Gen5 | ~3.938 | ~63 | ~126 |



T4 在 Gen3 x16 下，CPU→GPU 单向理论峰值约 **15.75 GB/s**（2026-11-07 在 Colab 上用 PyTorch 从 pinned 内存拷贝实测约 12.4 GB/s；普通 pageable 内存只有约 5 GB/s）。对比 T4 的显存带宽 **320 GB/s**：**数据一旦进了显存，访问速度比从 CPU 搬快 20 倍以上**。这就是为什么 GPU 编程要尽量减少“CPU↔GPU 来回搬数据”（[Lab 08](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/) 会再遇到）。

把这些数字按同一比例画出来，差距更直观：

![带宽阶梯：T4 显存带宽与各代 PCIe 理论带宽、Colab 实测拷贝带宽的对比](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/bandwidth-ladder.svg)

## 第 4 步：NUMA 归属（理解它，跳过实现）

前三步看的是“GPU 和 CPU 之间那条线”，这一步看“GPU 挂在哪颗 CPU 下面”。先跑两条命令，再逐个解释输出背后的概念：

In [ ]:
!lscpu | grep -E "^(Socket|NUMA)"
!nvidia-smi topo -m

在 Colab T4 上的实际输出如下（2026-11-07 实测，`topo -m` 末尾的 Legend 省略）：

```text
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0,1
    GPU0	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	0-1	0		N/A
```

### NUMA 是什么

**非一致内存访问（NUMA，Non-Uniform Memory Access）** 说的是：多路服务器上，每颗 CPU 都有自己的内存控制器，直连一组内存条。CPU 访问自己那组内存（本地内存）最快；要访问另一颗 CPU 的内存（远端内存），请求得经过 CPU 之间的互连（Intel 叫 UPI，AMD 叫 Infinity Fabric），多一跳，延迟更高、带宽受互连限制。“一颗 CPU + 它的本地内存”就构成一个 **NUMA 节点**。

PCIe 设备也一样：每颗 CPU 里有自己的 PCIe 根复合体，GPU、网卡插在哪颗 CPU 的 PCIe 通道上，就归属哪个 NUMA 节点。下图是一台双路服务器，GPU 0 读内存 0 走本地路径，读内存 1 则要跨过 CPU 互连：

![NUMA 是什么：双路服务器上每颗 CPU 带着自己的本地内存和 PCIe 设备，跨节点访问要经过 CPU 互连](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/numa-anatomy.svg)

对 GPU 来说，NUMA 影响的主要是“喂数据”这一段：训练进程在 CPU 上解码、预处理数据，再经 PCIe 拷进显存。如果进程跑在 CPU 1 上、数据放在内存 1，却要拷给挂在 CPU 0 下的 GPU 0，每次拷贝都得多绕一次互连。

### NUMA 是 NVIDIA 独有的吗

不是。NUMA 是 CPU 和内存的体系结构概念，与 GPU 厂商无关；GPU 只是 NUMA 拓扑里的一个“住户”。几个常见的疑问可以这样回答：



| 问题 | 回答 |
| :--- | :--- |
| 谁有 NUMA | 多路 Intel Xeon、AMD EPYC、Arm 服务器都是 NUMA 架构；单路小机器通常只有 1 个节点 |
| GPU 实现了 NUMA 吗 | 没有。GPU 是一个 PCIe 设备，它的 NUMA 归属由“插在哪颗 CPU 的 PCIe 通道上”决定，主板布线时就定了 |
| 其他加速卡呢 | 同样适用：AMD GPU（`rocm-smi --showtopo` 也会列出 NUMA 节点）、各类 AI 加速卡、高速网卡、NVMe 盘都有 NUMA 归属 |
| 一颗 CPU 就是一个 NUMA 节点吗 | 通常是，但不绝对。AMD EPYC 的 NPS（Nodes Per Socket）和 Intel 的 SNC（Sub-NUMA Clustering）可以把一颗 CPU 切成多个 NUMA 节点 |
| 显存会成为 NUMA 节点吗 | 普通 PCIe 卡不会。只有 Grace Hopper 这类 CPU 与 GPU 共享一致性内存的平台，才把显存暴露成独立 NUMA 节点，这就是 `GPU NUMA ID` 列的用途 |



### GPU 怎么知道自己在哪个 NUMA 节点

GPU 自己并不“探测” NUMA。开机时，主板固件通过 ACPI 表向操作系统上报拓扑：SRAT（System Resource Affinity Table）说明每颗 CPU、每段内存属于哪个邻近域，PCIe 根桥上的 `_PXM` 方法说明这座桥属于哪个域。Linux 内核把邻近域映射成 NUMA 节点，并给每个 PCI 设备记下 `numa_node`（所属节点）和 `local_cpulist`（最近的 CPU 核），通过 sysfs 暴露出来。`lscpu`、`nvidia-smi` 和 Kubernetes 的 kubelet 都是读取这些信息的“下游”：

![GPU 的 NUMA 归属从哪来：固件通过 ACPI 上报，Linux 内核记录到 sysfs，lscpu、nvidia-smi、kubelet 读取](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/numa-discovery.svg)

可以直接读出内核给这张 T4 记录的值（路径里的 `0000:00:04.0` 就是第 1 步看到的总线地址）：

In [ ]:
!cat /sys/bus/pci/devices/0000:00:04.0/numa_node
!cat /sys/bus/pci/devices/0000:00:04.0/local_cpulist

Colab 上的输出（2026-11-07 实测）：

```text
0
0-1
```

`0` 表示 T4 归属 NUMA 节点 0，`0-1` 表示离它最近的是 0、1 号 CPU 核，和下面 `topo -m` 里的 `NUMA Affinity`、`CPU Affinity` 完全一致：二者都来自内核记录的同一份拓扑信息。如果固件没有上报拓扑，`numa_node` 会是 `-1`。在多节点机器上，`/sys/devices/system/node/node0/distance` 还会给出到各节点的相对访问距离：本地是 `10`，远端常见 `20` 左右。

### 读懂 Colab 上的输出

有了上面的概念，回头看第 4 步开头的输出：前三行来自 `lscpu`，描述 CPU 侧的布局；后两行是 `nvidia-smi topo -m` 的拓扑矩阵，描述 GPU 挂在哪里。逐项含义如下：



| 字段 | Colab 上的值 | 含义 |
| :--- | :--- | :--- |
| `Socket(s)` | `1` | 物理 CPU 插槽数。多路服务器常见 `2` |
| `NUMA node(s)` | `1` | NUMA 节点数。只有 1 个时，所有内存对所有 CPU 一样近，不存在“远端内存” |
| `NUMA node0 CPU(s)` | `0,1` | 节点 0 包含的逻辑 CPU 编号。Colab 只分到 2 个 vCPU |
| 矩阵的行列 `GPU0` | `X` | 矩阵记录 GPU 两两之间的连接类型；对角线是自己，用 `X` 表示。单卡时矩阵只有这一格 |
| `CPU Affinity` | `0-1` | 离这张 GPU 最近的 CPU 核（与 sysfs 里的 `local_cpulist` 一致）。给 GPU 喂数据的进程最好绑在这些核上 |
| `NUMA Affinity` | `0` | GPU 归属的 NUMA 节点，即它“挂在”哪组 CPU 和内存下面 |
| `GPU NUMA ID` | `N/A` | 只有显存本身被暴露成 NUMA 节点时才有值；普通 PCIe 卡为 `N/A` |



所以 Colab 这台机器的结论很简单：只有 1 个 NUMA 节点，2 个 vCPU 和 T4 都在节点 0 上，不存在“选错节点”的问题。

### 多卡时的连接类型

单卡时矩阵里只有 `X`，Legend 里列出的连接类型都用不上。到了多卡服务器，矩阵的非对角格会填上这些缩写，表示两张卡之间的数据要走哪条路。下表按从近到远排列：



| 缩写 | 路径 | 通俗理解 |
| :--- | :--- | :--- |
| `NV#` | 走 # 条捆绑的 NVLink | GPU 直连，最快，不经过 PCIe |
| `PIX` | 最多经过一个 PCIe 桥 | 两张卡挂在同一个 PCIe Switch 下 |
| `PXB` | 经过多个 PCIe 桥，但不经过 Host Bridge | 跨了几级 Switch，仍不用绕到 CPU |
| `PHB` | 经过 PCIe Host Bridge（通常就是 CPU） | 数据要绕到 CPU 的根复合体再下来 |
| `NODE` | 经过同一 NUMA 节点内多个 Host Bridge 之间的互连 | 同一颗 CPU 下，但挂在不同根端口 |
| `SYS` | 经过 NUMA 节点之间的 SMP 互连（如 Intel QPI/UPI） | 跨 CPU 插槽，最慢 |



越靠上的连接，GPU 之间点对点通信（如 NCCL 的 all-reduce）越快。

### 拓扑感知调度是什么

把上面两件事合起来，就是 **拓扑感知调度（Topology-Aware Scheduling）**：分配资源时不只数“有几张卡、几个核”，还要看它们在拓扑里的位置，尽量让同一个任务用到的 GPU 彼此“近”（`NV#` / `PIX` 优于 `SYS`），并让 CPU 核、内存和 GPU 落在同一个 NUMA 节点。下图是一台四卡双路服务器，同样申请 2 张卡，三种选法的差别：

![拓扑感知分配：四卡双路服务器上，GPU 0 + GPU 1 走 NVLink 最优，GPU 2 + GPU 3 同一 Switch 可接受，GPU 1 + GPU 2 跨 CPU 互连应避免](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/topology-aware-allocation.svg)

在 Kubernetes 里，这件事主要由节点上的 kubelet 完成：

- **Topology Manager**：kubelet 的组件，收集各资源管理器给出的 NUMA“提示”（hint），为容器选一个大家都能满足的 NUMA 组合；
- **提示的来源**：CPU Manager（`static` 策略下给 Guaranteed Pod 分配独占核）、Memory Manager、Device Manager（Device Plugin 在上报每张 GPU 时附带它的 NUMA 节点，NVIDIA Device Plugin 还会按 NVLink 拓扑给出优选的卡组合）。

Topology Manager 的策略决定“对不齐时怎么办”：



| 策略 | 行为 |
| :--- | :--- |
| `none` | 默认值，不做拓扑对齐 |
| `best-effort` | 尽量对齐，对不齐也照样运行 |
| `restricted` | 对不齐就拒绝该 Pod（`TopologyAffinityError`） |
| `single-numa-node` | 要求所有资源来自同一个 NUMA 节点，否则拒绝 |



需要注意，默认的 kube-scheduler 看不到 NUMA 细节：它只知道某个节点“还有 2 张卡”，把 Pod 派过去后，才由 kubelet 判断能否对齐，严格策略下可能直接拒绝。想在调度阶段就考虑 NUMA，需要额外部署 scheduler-plugins 里的 NodeResourceTopology 插件，或者使用能描述设备属性和约束的 DRA。

> 这正是 Kubernetes [Device Plugin / DRA 边界](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/foundations/device-plugin-vs-dra/) 里“拓扑约束无法用整数表达”的现实来源：调度器不只关心“有几张卡”，还关心“卡和 CPU 谁更近”。

## 验证

完成后你应该能：

- 用 `lspci` 找到 GPU 的 bus:device.function；
- 把它和 `nvidia-smi` 的 `Bus Id` 对应起来；
- 说出 T4 的 PCIe 代际、链路宽度、理论带宽，以及为什么空载时 `current` 会低于 `max`；
- 解释为什么“数据进显存后比从 CPU 搬快 20 倍”；
- 说清 NUMA 是 CPU 和内存的概念，GPU 的 NUMA 归属来自固件和内核，以及拓扑感知调度在对齐什么。

## 故障模式



| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `lspci: command not found` | 镜像没有 `pciutils` | `!apt-get install -y -qq pciutils` |
| `lspci` 里看不到 NVIDIA | Runtime 还是 CPU | 切到 T4 GPU runtime，重连 |
| `pcie.link.gen.current` 是 `1` | 空载省电降速，正常 | 跑一个显存拷贝后再查，会升到 `3` |
| `pcie.link.width.current` 显示 `4` / `8` 而非 `16` | 虚拟化/共享带宽或降速 | 一般不影响学习；多见于被 vGPU/分片的环境 |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。